## SPARCS Hospital Inpatient Discharges (2024)

Dataset: https://health.data.ny.gov/d/sf4k-39ay

API documentation: https://dev.socrata.com/foundry/health.data.ny.gov/sf4k-39ay

### Imports

In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path

### Point to the API endpoint

The dataset page ID `sf4k-39ay` maps to a Socrata (SODA) API endpoint. Requesting `.json` from it returns records as a list of dictionaries, which pandas can turn into a DataFrame directly.

**App token (optional):** if `APP_TOKEN` is set in the `.env` file, it is sent with every request as the `X-App-Token` header, which avoids throttling. If it is empty or missing, the requests are sent without it and still work.

In [2]:
BASE_URL = "https://health.data.ny.gov/resource/sf4k-39ay.json"

load_dotenv()  # reads APP_TOKEN from the .env file at the repo root if it's set.

app_token = os.environ.get("APP_TOKEN")
HEADERS = {"X-App-Token": app_token} if app_token else {}

### Fetch Raw Data 

- With over 2 million rows, we request `PAGE_SIZE` rows at a time, moving the `$offset` forward each time. We order by `:id` (a built-in unique row id) so the pages are stable and don't overlap or skip rows.
- Downloading 2.1 million rows takes a while, so we save the result to a parquet file format (`data/raw/sparcs_2024_raw.parquet`) the first time. On every later run, the notebook reads that file from disk instead of calling the API.

To force a fresh download, delete the parquet file saved locally.

In [3]:
PAGE_SIZE = 50000

In [4]:
def fetch_page(offset):
    payload = {"$limit": PAGE_SIZE, "$offset": offset, "$order": ":id"}
    resp = requests.get(BASE_URL, params=payload, timeout=120, headers=HEADERS)
    resp.raise_for_status()
    return resp.json()

In [5]:
CACHE_PATH = Path("../data/raw/sparcs_2024_raw.parquet")
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

In [6]:
if CACHE_PATH.exists():
    df = pd.read_parquet(CACHE_PATH)
    print("loaded from cache")
else:
    pages = []
    offset = 0
    while True:
        page = fetch_page(offset)
        if not page:  # an empty page means we have gone past the last row
            break
        pages.append(page)
        print(f"fetched rows {offset:,} to {offset + len(page):,}")
        offset += PAGE_SIZE
    df = pd.DataFrame([row for page in pages for row in page])
    df.to_parquet(CACHE_PATH)
    print("downloaded and saved to cache.")

loaded from cache


In [7]:
### Sanity Check
count_resp = requests.get(BASE_URL, 
                    params={"$select":"count(*)"},
                    headers = HEADERS)
count_resp.raise_for_status()
total_rows = int(count_resp.json()[0]["count"])

if len(df) == total_rows:
    print(f"Row count matches the API: {len(df):,} rows")
else:
    print(f"WARNING: row count mismatch. API has {total_rows:,} rows, DataFrame has {len(df):,}")

Row count matches the API: 2,196,737 rows


In [8]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 2,196,737
Columns: 33
